In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

# =========================================
# Univariate logistic regressions (ORs)
# Full sample (1,321), except actigraphy
# predictors which restrict to available cases
# Includes FDR correction (BH)
# =========================================

in_path = "../data/mcs_sh_sample_Feb_28_OR_preprocessed_FULLSAMPLE.csv"
out_all = "../results/or/univariate_odds_ratios_95CI_full_sample_ALL.csv"
out_sig = "../results/or/univariate_odds_ratios_95CI_full_sample_SIG.csv"

target_col = "suicide_17y"
do_scale = True  # OR per 1 SD for numeric predictors

df = pd.read_csv(in_path)
df.columns = df.columns.str.strip()

df = df.replace(["", " ", "NA", "N/A", "null", "nan"], np.nan)

# Target mapping
df[target_col] = df[target_col].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
df = df.dropna(subset=[target_col]).copy()

# Features
X_full = df.drop(columns=[target_col]).copy()
y_full = df[target_col].astype(int)

# Enforce reference category ordering for dummy coding
if "child_alcohol" in X_full.columns:
    X_full["child_alcohol"] = pd.Categorical(X_full["child_alcohol"], ["none","some","many"])

if "child_cannabis" in X_full.columns:
    X_full["child_cannabis"] = pd.Categorical(X_full["child_cannabis"], ["Never","one to four","more than 5"])

if "mAlcohol_binary" in X_full.columns:
    X_full["mAlcohol_binary"] = pd.Categorical(X_full["mAlcohol_binary"], ["low risk","high risk"])

if "fAlcohol_binary" in X_full.columns:
    X_full["fAlcohol_binary"] = pd.Categorical(X_full["fAlcohol_binary"], ["low risk","high risk"])

if "mEdu" in X_full.columns:
    X_full["mEdu"] = pd.Categorical(X_full["mEdu"], ["lower edu","higher edu","overseas"])

# One hot encode remaining categoricals
cat_cols = X_full.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if len(cat_cols) > 0:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=True)

# Force numeric, handle inf (leave NaN alone here — handled per column below)
X_full = X_full.apply(pd.to_numeric, errors="coerce")
X_full = X_full.replace([np.inf, -np.inf], np.nan)

scale_cols = [c for c in X_full.columns if X_full[c].nunique(dropna=True) > 5]

results = []

for col in X_full.columns:
    # Build this predictor's own analysis frame, dropping rows missing
    # either the predictor or the outcome (per-column, not global)
    frame = pd.concat([X_full[[col]], y_full], axis=1).dropna()

    if frame[col].nunique(dropna=False) <= 1 or len(frame) == 0:
        continue

    x = frame[[col]].copy()
    endog = frame[target_col].astype(np.float64).to_numpy()

    if do_scale and col in scale_cols:
        scaler = StandardScaler()
        x[col] = scaler.fit_transform(x[[col]])

    exog = sm.add_constant(x, has_constant="add").to_numpy(dtype=np.float64)

    try:
        model = sm.Logit(endog, exog).fit(disp=0, maxiter=200)

        beta = model.params[1]
        se = model.bse[1]
        pval = model.pvalues[1]

        ci_low = beta - 1.96 * se
        ci_high = beta + 1.96 * se

        results.append({
            "Feature": col,
            "Beta": float(beta),
            "OR": float(np.exp(beta)),
            "CI_low": float(np.exp(ci_low)),
            "CI_high": float(np.exp(ci_high)),
            "p_value": float(pval),
            "N": int(len(endog)),
            "Error": ""
        })

    except Exception as e:
        results.append({
            "Feature": col,
            "Beta": np.nan,
            "OR": np.nan,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "p_value": np.nan,
            "N": int(len(endog)),
            "Error": str(e)
        })

or_table = pd.DataFrame(results)

# FDR correction (BH), only on valid p values
valid_mask = or_table["p_value"].notna()
pvals = or_table.loc[valid_mask, "p_value"].values

if len(pvals) > 0:
    _, pvals_fdr, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
    or_table.loc[valid_mask, "p_value_fdr"] = pvals_fdr
    or_table.loc[valid_mask, "significant_fdr"] = pvals_fdr < 0.05
else:
    or_table["p_value_fdr"] = np.nan
    or_table["significant_fdr"] = False

or_table_sorted = or_table.sort_values("p_value", ascending=True, na_position="last")
or_table_sorted.to_csv(out_all, index=False)

sig = or_table_sorted[or_table_sorted["significant_fdr"] == True].copy()
sig.to_csv(out_sig, index=False)

print(or_table_sorted[["Feature", "OR", "p_value", "p_value_fdr", "N"]].head(40))
print(f"Saved full univariate OR table: {out_all}")
print(f"Saved FDR significant univariate OR table: {out_sig}")